# SONIC fine-tuning walkthrough

이 노트북은 `s_batido` 모션을 Unitree G1용 SONIC 정책에 파인튜닝하는 전 과정을 단계별로 보여줍니다. 먼저 JupyterLab 오른쪽 위 커널을 **SONIC (Python 3.11)** 로 선택하세요.

긴 학습은 노트북 연결이 끊겨도 계속 돌도록 `tmux`에서 실행합니다.

## 1. GPU와 학습 환경 확인

In [ ]:
import subprocess
import sys
from pathlib import Path

import torch
import isaaclab

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=True)

## 2. 경로와 원본 모션 확인

In [ ]:
PROJECT = Path('/workspace/ultimate-bots-G1')
SONIC = Path('/workspace/GR00T-WholeBodyControl')
SOURCE_MOTION = PROJECT / 'data/source/sonic/s_batido_test_sonic'
MOTION_LIB = PROJECT / 'data/motion_lib/s_batido_test.pkl'
CHECKPOINT = SONIC / 'sonic_release/last.pt'

required_files = [SOURCE_MOTION / name for name in ('joint_pos.csv', 'body_pos.csv', 'body_quat.csv')]
print('Studio SONIC motion files:', required_files)
assert all(path.exists() for path in required_files), 'Studio에서 변환한 SONIC CSV 묶음이 필요합니다.'

## 3. SONIC motion-lib PKL로 변환

Studio가 만든 G1 관절·몸체 CSV 묶음을 SONIC이 읽는 motion-lib PKL 형식으로 변환합니다. Studio 출력 메타데이터 기준 50 FPS, 81프레임입니다.

In [ ]:
convert_cmd = [
    sys.executable,
    str(SONIC / 'gear_sonic/data_process/convert_soma_csv_to_motion_lib.py'),
    '--input', str(SOURCE_MOTION),
    '--output', str(MOTION_LIB),
    '--fps', '50',
]
print(' '.join(convert_cmd))
subprocess.run(convert_cmd, cwd=SONIC, check=True)
print('Generated PKL:', MOTION_LIB, MOTION_LIB.stat().st_size, 'bytes')

## 4. 공개 SONIC 체크포인트 받기

In [ ]:
download_cmd = [sys.executable, 'download_from_hf.py', '--training', '--no-smpl']
subprocess.run(download_cmd, cwd=SONIC, check=True)
assert CHECKPOINT.exists(), f'Checkpoint not found: {CHECKPOINT}'
print('Base checkpoint:', CHECKPOINT, CHECKPOINT.stat().st_size / 1024**2, 'MiB')

## 5. Before: 기본 정책 평가

파인튜닝 전 정책이 이 동작에서 얼마나 실패하는지 수치와 영상으로 남깁니다.

In [ ]:
before_dir = PROJECT / 'videos/before'
before_dir.mkdir(parents=True, exist_ok=True)
before_cmd = [
    sys.executable, 'gear_sonic/eval_agent_trl.py',
    f'+checkpoint={CHECKPOINT}',
    '+headless=True',
    '++eval_callbacks=im_eval',
    '++run_eval_loop=False',
    '++num_envs=1',
    '++manager_env.config.render_results=True',
    f'++manager_env.config.save_rendering_dir={before_dir}',
    '~manager_env/recorders=empty',
    '+manager_env/recorders=render',
    f'++manager_env.commands.motion.motion_lib_cfg.motion_file={MOTION_LIB}',
    '++manager_env.commands.motion.motion_lib_cfg.smpl_motion_file=dummy',
]
print(' '.join(map(str, before_cmd)))
# 환경 설치와 데이터 변환을 확인한 뒤 아래 주석을 해제합니다.
# subprocess.run(before_cmd, cwd=SONIC, env={'ACCEPT_EULA': 'Y', **dict(__import__('os').environ)}, check=True)

## 6. 짧은 파인튜닝 smoke test

먼저 16개 환경에서 5 iteration만 실행해 전체 파이프라인에 오류가 없는지 확인합니다.

In [ ]:
smoke_cmd = [
    sys.executable, 'gear_sonic/train_agent_trl.py',
    '+exp=manager/universal_token/all_modes/sonic_release',
    f'+checkpoint={CHECKPOINT}',
    'num_envs=16',
    'headless=True',
    '++algo.config.num_learning_iterations=5',
    f'++manager_env.commands.motion.motion_lib_cfg.motion_file={MOTION_LIB}',
    '++manager_env.commands.motion.motion_lib_cfg.smpl_motion_file=dummy',
    'use_wandb=false',
]
print(' '.join(map(str, smoke_cmd)))
# subprocess.run(smoke_cmd, cwd=SONIC, env={'ACCEPT_EULA': 'Y', **dict(__import__('os').environ)}, check=True)

## 7. 본 파인튜닝 시작

학습은 노트북을 닫아도 계속 실행되도록 `tmux` 세션에서 시작합니다. 처음에는 A40에 무리가 없는 1024 environments로 시작하고 GPU 메모리를 본 뒤 조정합니다.

In [ ]:
train_command = ' '.join([
    sys.executable, 'gear_sonic/train_agent_trl.py',
    '+exp=manager/universal_token/all_modes/sonic_release',
    f'+checkpoint={CHECKPOINT}',
    'num_envs=1024', 'headless=True',
    '++algo.config.actor_learning_rate=5e-6',
    '++algo.config.desired_kl=0.005',
    f'++manager_env.commands.motion.motion_lib_cfg.motion_file={MOTION_LIB}',
    '++manager_env.commands.motion.motion_lib_cfg.smpl_motion_file=dummy',
    'wandb.wandb_project=ultimate-bots-g1',
])
log_file = PROJECT / 'train.log'
tmux_cmd = [
    'tmux', 'new-session', '-d', '-s', 'sonic_train',
    f"cd {SONIC} && ACCEPT_EULA=Y {train_command} 2>&1 | tee {log_file}",
]
print(' '.join(tmux_cmd))
# subprocess.run(tmux_cmd, check=True)

In [ ]:
# 최근 학습 로그 확인
if log_file.exists():
    print(''.join(log_file.read_text(errors='replace').splitlines(True)[-60:]))
else:
    print('아직 train.log가 없습니다.')

## 8. After 평가 및 ONNX export

학습 후에는 마지막 파일이 아니라 평가 점수가 가장 좋은 체크포인트를 `BEST_CHECKPOINT`로 지정합니다.

In [ ]:
BEST_CHECKPOINT = Path('/workspace/GR00T-WholeBodyControl/logs_rl/TRL_G1_Track/CHANGE_ME/model_step_002000.pt')
export_cmd = [
    sys.executable, 'gear_sonic/eval_agent_trl.py',
    f'+checkpoint={BEST_CHECKPOINT}',
    '+headless=True', '++num_envs=1', '+export_onnx_only=true',
]
print(' '.join(map(str, export_cmd)))
# subprocess.run(export_cmd, cwd=SONIC, env={'ACCEPT_EULA': 'Y', **dict(__import__('os').environ)}, check=True)